# 02. Composição da base

Como a base do notebook 01 foi montada para bater com o cenário:

- **Leads**: ~280k em 180 dias, com menos entrada no fim de semana e pico em julho e novembro.
- **Canal nulo (~25%)**: orgânico e e-mail chegam sem utm com mais frequência, então o nulo não é aleatório.
- **Abordagem**: o comercial trabalha em dias úteis, das 9h às 18h. Lead do fim de semana, de feriado ou de depois das 18h entra na fila do próximo dia útil, e em cada dia são abordados os primeiros 20% da fila.
- **Matrícula (~3% nos abordados)**: depende do perfil do lead e de ele ter sido atendido. Matrícula depois da data de corte não entra.
- **Navegação (~40% com lead_id)**: antes do cadastro a navegação é anônima; a sessão do cadastro liga as duas partes pelo `session_id`.

In [0]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, IntegerType

FQ = "cogna.captacao"
DT_FIM = "2026-11-22"
FERIADOS = ["2026-06-04", "2026-09-07", "2026-10-12", "2026-11-02", "2026-11-15", "2026-11-20"]

leads = spark.table(f"{FQ}.tb_leads")
crm = spark.table(f"{FQ}.tb_abordagem_crm")
nav = spark.table(f"{FQ}.tb_navegacao")
spark.sql("USE cogna.captacao")

# calendário: tipo do dia e dia da fila (próximo dia útil)
ini, fim = date(2026, 5, 1), date(2027, 1, 31)
dias = [ini + timedelta(days=i) for i in range((fim - ini).days + 1)]
linhas, prox = [], None
for d in reversed(dias):
    tipo = "feriado" if d.isoformat() in FERIADOS else ("fim_de_semana" if d.weekday() >= 5 else "dia_util")
    prox_depois = prox
    if tipo == "dia_util":
        prox = d
    linhas.append((d, tipo, prox, prox_depois))
spark.createDataFrame(linhas, "dt date, tipo_dia string, prox_dia_util date, prox_dia_util_depois date").createOrReplaceTempView("vw_calendario")

## 1. Volumes
Quantidade de linhas e período coberto por tabela.

In [0]:
def resumo(df, nome, col_data):
    return df.agg(F.lit(nome).alias("tabela"), F.count("*").alias("linhas"),
                  F.min(col_data).alias("inicio"), F.max(col_data).alias("fim"))

display(resumo(leads, "tb_leads", "dt_criacao")
        .unionByName(resumo(crm, "tb_abordagem_crm", "dt_tentativa_contato"))
        .unionByName(resumo(nav, "tb_navegacao", "ts_evento")))

## 2. Leads
Entrada ao longo do tempo, perfil (curso, modalidade, canal), nulos e matrícula.

In [0]:
display(leads.limit(20))

In [0]:
# nulos por coluna
display(leads.select([F.round(F.avg(F.col(c).isNull().cast(IntegerType())) * 100, 2).alias(c) for c in leads.columns]))

In [0]:
%sql
-- leads por dia
SELECT to_date(dt_criacao) AS dia, count(*) AS leads
FROM tb_leads
GROUP BY 1
ORDER BY 1

In [0]:
%sql
-- média de leads por tipo de dia
WITH por_dia AS (
  SELECT to_date(dt_criacao) AS dt, count(*) AS leads FROM tb_leads GROUP BY 1
)
SELECT c.tipo_dia, count(*) AS dias, round(avg(p.leads)) AS media_leads
FROM por_dia p JOIN vw_calendario c USING (dt)
GROUP BY 1
ORDER BY 3 DESC

In [0]:
%sql
SELECT hour(dt_criacao) AS hora, count(*) AS leads
FROM tb_leads
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT curso, modalidade, leads, round(100 * leads / leads_curso, 1) AS pct_no_curso
FROM (
  SELECT curso, modalidade, count(*) AS leads, sum(count(*)) OVER (PARTITION BY curso) AS leads_curso
  FROM tb_leads
  GROUP BY 1, 2
)
ORDER BY leads_curso DESC, curso, modalidade

In [0]:
%sql
SELECT coalesce(canal_origem, 'vazio') AS canal_origem, count(*) AS leads,
       round(100 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM tb_leads
GROUP BY 1
ORDER BY 2 DESC

In [0]:
%sql
-- matrícula por mês de entrada (meses recentes ainda não tiveram tempo de converter)
SELECT date_trunc('month', dt_criacao) AS mes, count(*) AS leads, sum(fl_matricula) AS matriculas,
       round(100 * avg(fl_matricula), 2) AS pct_matricula
FROM tb_leads
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT datediff(dt_matricula, to_date(dt_criacao)) AS dias_ate_matricula, count(*) AS matriculas
FROM tb_leads
WHERE fl_matricula = 1
GROUP BY 1
ORDER BY 1

## 3. Abordagem CRM
Fila por dia útil, cobertura, tentativas, status e matrícula de abordados x não abordados.

In [0]:
display(crm.orderBy("lead_id", "dt_tentativa_contato").limit(20))

In [0]:
%sql
-- uma linha por lead: resumo do CRM e dia da fila (próximo dia útil quando entra fora do expediente)
CREATE OR REPLACE TEMP VIEW vw_leads_crm AS
WITH crm_lead AS (
  SELECT lead_id,
         count(*) AS qtd_tentativas,
         min(dt_tentativa_contato) AS dt_primeira,
         max_by(status_contato, dt_tentativa_contato) AS status_final
  FROM tb_abordagem_crm
  GROUP BY lead_id
)
SELECT l.*,
       c.qtd_tentativas, c.dt_primeira, c.status_final,
       CASE WHEN cal.tipo_dia <> 'dia_util' THEN cal.prox_dia_util
            WHEN hour(l.dt_criacao) >= 18 THEN cal.prox_dia_util_depois
            ELSE cal.dt END AS dia_fila,
       CASE WHEN c.lead_id IS NULL THEN 0 ELSE 1 END AS abordado
FROM tb_leads l
LEFT JOIN crm_lead c ON c.lead_id = l.lead_id
JOIN vw_calendario cal ON cal.dt = to_date(l.dt_criacao)

In [0]:
%sql
-- tentativas por tipo de dia: o comercial não trabalha no fim de semana nem em feriado
SELECT c.tipo_dia, count(*) AS tentativas
FROM tb_abordagem_crm t JOIN vw_calendario c ON to_date(t.dt_tentativa_contato) = c.dt
GROUP BY 1

In [0]:
%sql
-- tamanho da fila e % abordado por dia útil (segunda acumula o fim de semana)
SELECT dia_fila, date_format(dia_fila, 'E') AS dia_semana, count(*) AS fila, sum(abordado) AS abordados,
       round(100 * avg(abordado), 1) AS pct_abordados
FROM vw_leads_crm
WHERE dia_fila <= '2026-11-22'
GROUP BY 1, 2
ORDER BY 1

In [0]:
%sql
-- % abordado por hora de entrada
SELECT hour(dt_criacao) AS hora, count(*) AS leads, round(100 * avg(abordado), 1) AS pct_abordados
FROM vw_leads_crm
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT qtd_tentativas, status_final, count(*) AS leads
FROM vw_leads_crm
WHERE abordado = 1
GROUP BY 1, 2
ORDER BY 1, 3 DESC

In [0]:
%sql
SELECT canal_contato, status_contato, count(*) AS tentativas,
       round(100 * count(*) / sum(count(*)) OVER (PARTITION BY canal_contato), 1) AS pct_no_canal
FROM tb_abordagem_crm
GROUP BY 1, 2
ORDER BY 1, 3 DESC

In [0]:
%sql
SELECT CASE WHEN abordado = 1 THEN 'abordado' ELSE 'nao_abordado' END AS grupo,
       count(*) AS leads, sum(fl_matricula) AS matriculas, round(100 * avg(fl_matricula), 2) AS pct_matricula
FROM vw_leads_crm
GROUP BY 1

## 4. Navegação
Em produção essa tabela tem bilhões de linhas, então:
1. a classificação das sessões fica gravada em `tb_nav_sessao` e cada rodada classifica só as sessões novas; o que já foi classificado não é reprocessado;
2. as análises leem esse resumo e uma janela de tempo da tabela bruta (clusterizada por data, então o filtro lê só os arquivos do período);
3. distribuições em amostra e contagem distinta aproximada.

In [0]:
SESSOES = f"{FQ}.tb_nav_sessao"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SESSOES} (
  session_id STRING, lead_id STRING, inicio TIMESTAMP, fim TIMESTAMP,
  eventos BIGINT, eventos_ident BIGINT, tipo_sessao STRING, dt_classificacao TIMESTAMP
) CLUSTER BY (fim)
""")

# só lê eventos depois do último processado (com 2h de folga para sessões em andamento)
ultimo = spark.table(SESSOES).agg(F.max("fim").alias("fim")).collect()[0]["fim"]
corte = None if ultimo is None else ultimo - timedelta(hours=2)
eventos_novos = nav if corte is None else nav.filter(F.col("ts_evento") > F.lit(corte))
ja_classificadas = spark.table(SESSOES).select("session_id") if corte is None else \
    spark.table(SESSOES).filter(F.col("fim") > F.lit(corte)).select("session_id")

qtd_eventos, qtd_ident = F.count("*"), F.count("lead_id")
novas = (eventos_novos.groupBy("session_id")
    .agg(F.max("lead_id").alias("lead_id"),
         F.min("ts_evento").alias("inicio"),
         F.max("ts_evento").alias("fim"),
         qtd_eventos.alias("eventos"),
         qtd_ident.alias("eventos_ident"),
         F.when(qtd_ident == 0, "anonima").when(qtd_ident == qtd_eventos, "identificada").otherwise("mista").alias("tipo_sessao"),
         F.current_timestamp().alias("dt_classificacao"))
    .join(ja_classificadas, "session_id", "left_anti"))
novas.write.mode("append").saveAsTable(SESSOES)

display(spark.table(SESSOES).groupBy("dt_classificacao").count().orderBy("dt_classificacao"))

In [0]:
JANELA_DIAS = 90
inicio_janela = F.date_sub(F.lit(DT_FIM).cast(DateType()), JANELA_DIAS)
nav_janela = nav.filter(F.col("ts_evento") >= inicio_janela)
sessoes = spark.table(SESSOES).filter(F.col("inicio") >= inicio_janela)

display(nav_janela.filter("lead_id IS NOT NULL").limit(20))

In [0]:
qtd_leads = leads.count()
display(sessoes.agg(F.sum("eventos").alias("eventos"),
                    F.sum("eventos_ident").alias("eventos_ident"),
                    F.count("*").alias("sessoes"),
                    F.approx_count_distinct("lead_id").alias("leads_com_nav"),
                    F.round(100 * F.sum("eventos_ident") / F.sum("eventos"), 1).alias("pct_eventos_ident"),
                    F.round(100 * F.approx_count_distinct("lead_id") / F.lit(qtd_leads), 1).alias("pct_leads_com_nav")))

In [0]:
display(sessoes.groupBy("tipo_sessao")
        .agg(F.count(F.lit(1)).alias("sessoes"), F.sum("eventos").alias("eventos"),
             F.sum(F.col("eventos") - F.col("eventos_ident")).alias("eventos_sem_lead"))
        .orderBy(F.count(F.lit(1)).desc()))

In [0]:
# exemplo de sessão mista: antes do cadastro sem lead_id, depois com
ex = sessoes.filter("tipo_sessao = 'mista' AND eventos >= 5").select("session_id", "inicio").limit(1).collect()
if ex:
    inicio, sessao = ex[0]["inicio"], ex[0]["session_id"]
    display(nav.filter(F.col("ts_evento").between(inicio, inicio + timedelta(hours=2)) & (F.col("session_id") == sessao))
            .orderBy("ts_evento"))

In [0]:
sessoes_tamanho = sessoes.withColumn("eventos_na_sessao", F.least(F.col("eventos"), F.lit(30)))
display(sessoes_tamanho.groupBy("eventos_na_sessao").count().orderBy("eventos_na_sessao"))

In [0]:
# páginas: contagem lendo só a coluna pagina; tempo de permanência em amostra de 5%
display(nav_janela.groupBy("pagina").agg(F.count(F.lit(1)).alias("eventos")).orderBy(F.count(F.lit(1)).desc()))

In [0]:
amostra = nav_janela.sample(fraction=0.05, seed=1)
display(amostra.groupBy("pagina")
        .agg(F.percentile_approx("tempo_permanencia_seg", 0.5).alias("mediana_seg"),
             F.percentile_approx("tempo_permanencia_seg", 0.9).alias("p90_seg"))
        .orderBy("pagina"))

In [0]:
amostra_faixa = amostra.withColumn("faixa_seg", F.least(F.floor(F.col("tempo_permanencia_seg") / 15) * 15, F.lit(600)))
display(amostra_faixa.groupBy("faixa_seg").count().orderBy("faixa_seg"))

## 5. Integridade
Chaves órfãs, duplicidade e datas inconsistentes.

In [0]:
ids = leads.select("lead_id")
cal = spark.table("vw_calendario")
nao_util = cal.filter(cal["tipo_dia"] != "dia_util")
checks = [
    ("crm sem lead", crm.select("lead_id").join(ids, "lead_id", "left_anti").count()),
    ("navegacao sem lead", sessoes.filter("lead_id IS NOT NULL").select("lead_id").distinct().join(ids, "lead_id", "left_anti").count()),
    ("lead_id duplicado", qtd_leads - ids.distinct().count()),
    ("matricula antes da criacao", leads.filter(F.col("dt_matricula") < F.to_date("dt_criacao")).count()),
    ("tentativa fora de dia util", crm.join(F.broadcast(nao_util), F.to_date(crm["dt_tentativa_contato"]) == nao_util["dt"]).count()),
]
display(spark.createDataFrame(checks, "checagem string, ocorrencias long"))